# OpenPlaque RCA Source-CCTA Seeded Tracer — FINAL

Built from the clean `main`-derived branch. This notebook uses source CCTA **series 7** and never uses `%matplotlib widget`, `ipympl`, or any custom Matplotlib backend.

Choose **Runtime → Run all**. Google Drive mounts first. At the end, use the z/x/y sliders to place the crosshair in the RCA lumen, save an **OSTIUM** seed, move distally and save a **DISTAL** seed, then press **Trace RCA**.

Research prototype only. Visually verify every trace before quantitative use.

In [ ]:
# ALWAYS FIRST: mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')
print('Google Drive mounted.')


In [ ]:
# Fresh clone of the clean main-derived branch and standard dependencies only.
!rm -rf /content/OpenPlaque
!git clone -q --branch rca-centerline-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK scipy scikit-image matplotlib ipywidgets pytest

import os, sys, time, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

SRC = Path('/content/OpenPlaque/src')
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from openplaque.study import OpenPlaqueStudy
from openplaque.source_centerline import trace_seeded_coronary
print('OpenPlaque source:', SRC)
print('No interactive Matplotlib backend is used in this notebook.')


In [ ]:
# Quick synthetic unit test of the new tracer code.
!python -m pytest -q /content/OpenPlaque/tests/test_source_centerline.py


## Load source CCTA series 7
The DICOM ZIP is copied from Drive to local Colab storage before extraction/scanning.

In [ ]:
ROOT = Path('/content/drive/MyDrive/OpenPlaque')
DRIVE_ZIP = ROOT / 'Full_DICOM.zip'
LOCAL_ZIP = Path('/content/Full_DICOM.zip')
EXTRACT_ROOT = '/content/full_dicom_seeded_rca_final'
SOURCE_SERIES = 7

if not DRIVE_ZIP.exists():
    raise FileNotFoundError(f'Missing {DRIVE_ZIP}')

t = time.time()
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != DRIVE_ZIP.stat().st_size:
    print(f'Copying Full_DICOM.zip ({DRIVE_ZIP.stat().st_size/1e9:.2f} GB) to local disk...', flush=True)
    shutil.copyfile(DRIVE_ZIP, LOCAL_ZIP)
    print(f'Copy finished in {time.time()-t:.1f}s', flush=True)
else:
    print('Local ZIP already staged.')

shutil.rmtree(EXTRACT_ROOT, ignore_errors=True)
print('Extracting/scanning DICOM locally...', flush=True)
t = time.time()
study = OpenPlaqueStudy(str(LOCAL_ZIP), extract_root=EXTRACT_ROOT)
print(f'Scan finished in {time.time()-t:.1f}s; {len(study.series)} series found.', flush=True)
source_img, source, source_files = study.load_series(SOURCE_SERIES)
print('Loaded source series:', SOURCE_SERIES)
print('Shape zyx:', source.shape)
print('Spacing xyz mm:', source_img.GetSpacing())


## Seed picker and tracer
Use the sliders to move the crosshair. The left panel shows the full axial slice; the right panel is a zoom around the crosshair. Save the ostium and distal seed, then trace.

In [ ]:
state = {'ostium': None, 'distal': None, 'result': None}

z_slider = widgets.IntSlider(value=min(335, source.shape[0]-1), min=0, max=source.shape[0]-1, step=1, description='z', continuous_update=False, layout=widgets.Layout(width='700px'))
x_slider = widgets.IntSlider(value=source.shape[2]//2, min=0, max=source.shape[2]-1, step=1, description='x', continuous_update=False, layout=widgets.Layout(width='700px'))
y_slider = widgets.IntSlider(value=source.shape[1]//2, min=0, max=source.shape[1]-1, step=1, description='y', continuous_update=False, layout=widgets.Layout(width='700px'))
save_ostium = widgets.Button(description='Save as OSTIUM', button_style='success')
save_distal = widgets.Button(description='Save as DISTAL', button_style='info')
trace_button = widgets.Button(description='Trace RCA', button_style='warning')
status = widgets.HTML(value='<b>Move the crosshair to the RCA ostium.</b>')
viewer_out = widgets.Output()
trace_out = widgets.Output()

def current_seed():
    return (int(z_slider.value), int(y_slider.value), int(x_slider.value))

def render_view(*_):
    z, y, x = current_seed()
    r = 35
    y0, y1 = max(0, y-r), min(source.shape[1], y+r+1)
    x0, x1 = max(0, x-r), min(source.shape[2], x+r+1)
    with viewer_out:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(13, 6))
        axes[0].imshow(source[z], cmap='gray', vmin=-200, vmax=800)
        axes[0].axvline(x, linewidth=0.8); axes[0].axhline(y, linewidth=0.8)
        axes[0].set_title(f'Series 7 axial z={z}   crosshair x={x}, y={y}')
        axes[0].axis('off')
        axes[1].imshow(source[z, y0:y1, x0:x1], cmap='gray', vmin=-200, vmax=800)
        axes[1].axvline(x-x0, linewidth=0.8); axes[1].axhline(y-y0, linewidth=0.8)
        axes[1].set_title(f'Zoom   HU={float(source[z,y,x]):.0f}')
        axes[1].axis('off')
        plt.tight_layout(); plt.show(); plt.close(fig)
        print('Current zyx:', (z,y,x), 'HU:', float(source[z,y,x]))
        print('Saved ostium:', state['ostium'], '   Saved distal:', state['distal'])

for w in (z_slider, x_slider, y_slider):
    w.observe(render_view, names='value')

def save_seed(which):
    state[which] = current_seed()
    status.value = f'Saved <b>{which}</b> at zyx={state[which]}. Ostium={state["ostium"]}; distal={state["distal"]}'
    render_view()

save_ostium.on_click(lambda b: save_seed('ostium'))
save_distal.on_click(lambda b: save_seed('distal'))

def physical_xyz_to_zyx(pt_xyz):
    ix = source_img.TransformPhysicalPointToContinuousIndex(tuple(float(v) for v in pt_xyz))
    return np.array([ix[2], ix[1], ix[0]], dtype=float)

def show_trace(result):
    route = np.asarray(result.points_zyx_voxel, dtype=float)
    margin = np.ceil(12.0 / np.asarray(source_img.GetSpacing())[::-1]).astype(int)
    lo = np.maximum(0, np.floor(route.min(axis=0)).astype(int) - margin)
    hi = np.minimum(np.asarray(source.shape), np.ceil(route.max(axis=0)).astype(int) + margin + 1)
    crop = source[lo[0]:hi[0], lo[1]:hi[1], lo[2]:hi[2]]
    rr = route - lo
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    axes[0].imshow(np.max(crop, axis=0), cmap='gray', vmin=-100, vmax=700)
    axes[0].plot(rr[:,2], rr[:,1], '-', linewidth=2); axes[0].set_title('Axial projection')
    axes[1].imshow(np.max(crop, axis=1), cmap='gray', vmin=-100, vmax=700, aspect='auto')
    axes[1].plot(rr[:,2], rr[:,0], '-', linewidth=2); axes[1].set_title('Coronal projection')
    axes[2].imshow(np.max(crop, axis=2), cmap='gray', vmin=-100, vmax=700, aspect='auto')
    axes[2].plot(rr[:,1], rr[:,0], '-', linewidth=2); axes[2].set_title('Sagittal projection')
    for d, pt in sorted(result.landmarks_xyz_mm.items()):
        q = physical_xyz_to_zyx(pt) - lo
        axes[0].scatter([q[2]], [q[1]], s=70); axes[0].text(q[2]+2, q[1], f'{d:.0f} mm')
        axes[1].scatter([q[2]], [q[0]], s=70); axes[1].text(q[2]+2, q[0], f'{d:.0f} mm')
        axes[2].scatter([q[1]], [q[0]], s=70); axes[2].text(q[1]+2, q[0], f'{d:.0f} mm')
    for a in axes: a.axis('off')
    plt.tight_layout(); plt.show(); plt.close(fig)

def do_trace(_):
    if state['ostium'] is None or state['distal'] is None:
        status.value = '<b>Save both OSTIUM and DISTAL seeds first.</b>'
        return
    with trace_out:
        clear_output(wait=True)
        print('Tracing RCA with source-CCTA vesselness + shortest path...')
        t = time.time()
        try:
            result = trace_seeded_coronary(source, source_img, state['ostium'], state['distal'], crop_margin_mm=35.0)
            state['result'] = result
            print(f'Done in {time.time()-t:.1f}s. Centerline length = {result.length_mm:.1f} mm')
            print('Landmarks:', sorted(result.landmarks_xyz_mm))
            if 50.0 not in result.landmarks_xyz_mm:
                print('FAIL: trace is shorter than 50 mm; choose a more distal seed.')
            show_trace(result)
            print('PASS only if the line follows the RCA continuously and the 10/50-mm points are anatomically plausible.')
        except Exception as e:
            print('TRACE FAILED:', repr(e))

trace_button.on_click(do_trace)

display(z_slider, x_slider, y_slider, widgets.HBox([save_ostium, save_distal, trace_button]), status, viewer_out, trace_out)
render_view()
print('Run All complete. Use the controls above; no additional cells need to be run.')
